In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession \
    .builder \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .appName("ex3_clean_flights") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/01 11:52:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [12]:
flights_df = spark.read.parquet('s3a://pyspark/data/source/flights/')
flights_raw_df = spark.read.parquet('s3a://pyspark/data/source/flights_raw/', header=True)

In [13]:
flights_distinct_df = flights_df.dropDuplicates()
flights_raw_distinct_df = flights_raw_df.dropDuplicates()

In [ ]:
matched_df = flights_distinct_df.intersect(flights_raw_distinct_df)

unmatched_flights_df = flights_distinct_df.subtract(matched_df) \
    .withColumn("source_of_data", F.lit("flights"))

unmatched_flights_raw_df = flights_raw_distinct_df.subtract(matched_df) \
    .withColumn("source_of_data", F.lit("flights_raw"))

unmatched_df = unmatched_flights_df.union(unmatched_flights_raw_df)

In [19]:
matched_df.write.parquet('s3a://pyspark/data/stg/flights_matched/', mode='overwrite')
unmatched_df.write.parquet('s3a://pyspark/data/stg/flights_unmatched/', mode='overwrite')

In [20]:
spark.stop()